# Task 2
## Integrantes
* Sergio Orellana 221122
* Rodrigo Mansilla 22611
* Ricardo Chuy 221007

Las siguientes preguntas evalúan su comprensión estratégica de la evolución de los detectores de dos etapas y su capacidad de tomar decisiones de arquitectura justificadas dentro del contexto operativo de VisorShelf. Se valorará la coherencia del argumento con las restricciones reales del sistema.

---
## Pregunta 2.1

El CTO de VisorShelf propone usar el detector original **R-CNN (2014)** para la primera versión del sistema. El equipo de ingeniería calcula que con el dataset actual y una CPU de tienda, cada imagen tardaría aproximadamente **45 segundos** en procesarse.

Con esto responda en su reporte:

**1.** Identifique el cuello de botella principal de R-CNN que causa esa latencia. Explique por qué procesar 2,000 propuestas de región de forma independiente es computacionalmente costoso, conectando su respuesta con lo que la red hace internamente en cada pasada.

**2.** Fast R-CNN introdujo el **feature map compartido** y el **RoI Pooling** para resolver ese cuello de botella. Explique la lógica detrás de cada uno: ¿qué cómputo elimina el feature map compartido?, ¿qué problema resuelve el RoI Pooling y qué operación matemática realiza para producir un tensor de tamaño fijo a partir de regiones de tamaño variable?

**3.** Con Fast R-CNN el tiempo de CNN bajó a 0.3 s/imagen, pero el tiempo total seguía siendo ~2.3 s. Identifique el nuevo cuello de botella y explique por qué Selective Search representa un problema arquitectónico más profundo que simplemente ser lento.

### Respuesta 2.1.1

- Lo que pasa es que R-CNN al analizar y proponer 2000 regiones donde podría haber un objeto, introduce mucha complejidad y cálculos por cada una de esas regiones propuestas. Todas esas regiones se redimensionan a ser de 227x227 pixeles y aparte de eso cada región se pasa a través de una CNN, específicamente alexnet. Por cada región, la CNN produce un vector de características y luego un clasificador SVM decide que objeto es. Lo que pasa es que pasar 2000 propuestas en una CNN son muchísimas operaciones de convolución y encima de eso, puede que las 2000 regiones sean redundantes. Eso quiere decir que 2000 propuestas, puede que algunas cubran el mismo objeto. AlexNet tiene millones de parámetros, por lo que al final se hacen millones de operaciones por cada región propuesta.

### Respuesta 2.1.2

- El feature map compartido principalmente ayuda a eliminar las 2000 repeticiones que se hacían de R-CNN en la CNN. En cada celda del feature map tendríamos una región de la imagen original y cada celda tiene 512 canales que ayudan a describir una región. Pero la diferncia clave es que Fast R-CNN pasa la imagen solo 1 vez por la CNN para obtener un feature map. El problema ahora es que cada región proyectada sobre el feature map tiene un tamaño variable pero las capas fully-connected necesitan un tensor de tamaño fijo. Entonces en el caso de Rol Pooling soluciona incompatibilidad de tamaños, esto se resuelve esto dividiendo la región proyectada en una región de tamaño H*W y aplicando max pooling. Entonces el resultado siempre es un tensor H*W*C independientemente del tamaño original de la región.

### Respuesta 2.1.3

- Lo que pasa es que selective search es un algoritmo de visión por computadora clásico. Es decir fue antes de muchos de los desarrrollos y descubrimientos de deep learning. Sus reglas son fijas y siempre divide una imagen en distintas partes quse van fusionando basándose en similaidad de color, texturas y tamaño por ejemplo. No es un algoritmo de redes neuronales que aprenda o se adapte a los datos. El hecho de que el algoritmo y sus reglas sean fijas implica que la calidad de las propuestas de región nunca mejora aunque se tenga más data etiquetada o el modelo entrene por más tiempo. El sistema esta limitado por la capacidad de las regiones que propone selective search.

---
## Pregunta 2.2

El equipo de VisorShelf decide usar Faster R-CNN como base del sistema. Un ingeniero junior pregunta: *"¿Por qué necesitamos una Region Proposal Network si ya tenemos el feature map? ¿No podríamos simplemente hacer sliding window directamente sobre el feature map?"*

Responda en su reporte:

**4.** Responda la pregunta del ingeniero junior. Explique qué hace la RPN que un sliding window clásico no puede hacer, y por qué el hecho de que la RPN opere sobre el mismo feature map del backbone es una ventaja semántica, no solo computacional.

**5.** La RPN utiliza **anchor boxes** predefinidos (9 por posición: 3 escalas × 3 relaciones de aspecto). Explique qué son los anchors y por qué la red predice **deltas** (Δx, Δy, Δw, Δh) en lugar de coordenadas absolutas. En la decodificación **w = w_a · e^(Δw)**, identifique qué representa cada símbolo y explique por qué se usa la exponencial para las dimensiones.

**6.** Faster R-CNN logra ~5 FPS con VGG16. La restricción de VisorShelf es procesar una imagen en menos de **500 ms (≥ 2 FPS)**. Tomando en cuenta esa restricción, ¿recomendaría Faster R-CNN para producción en el hardware de tienda? Argumente su respuesta considerando al menos dos factores distintos a la velocidad pura (p. ej., precisión en objetos pequeños y densos, facilidad de fine-tuning, ecosistema de implementación).

### Respuesta 2.2.4
- Lo que pasa es que un sliding window clásico pasa una ventana de tamaño fijo sobre la imagen, pero lo hace varias veces cambiando las dimensiones de la ventana. Opera sobre los píxeles crudos sin aprendizazje con la ayuda de un clasifiicador para determinar si hay un objeto o no y el tamaño de la ventana va incrementando para encontrar objetos de distintos tamaños. Pero esto genera miles/millones de posiciones a evaluar. Entonces la gran diferencia y la razón por la que la RPN es mejor en general, es que RPN como bien dice el nombre es ubna red neuronal, que se desliza en el feature map del backbone. Como no es sobre los pixles, la red es capaz de obtener del feature map información semántica aprendida.  La RPN usa eso para proponer regiones de forma inteligente y encima se entrena junto con el detector, lo que hace que mejore conforme el modelo aprende. 


### Respuesta 2.2.5

- Los anchors son cajas de referencia que ya están predefinidas y distribuidas en el feature map. En Faster R-CNN hay 9 por posición debido a que hay 3 escalas: 128², 256², 512² px por 3 distinas proporciones de aspecto: 1:1, 1:2, 2:1. Esto se hace con el fin de cubrir el rango de tamaños y formas esperados de los objetos. La red predice deltas en lugar de coordennadas porque, la red predice ajuestes pequeños sobre las cajas/anchors de referencia en lugar de revisar todos los posibles pixeles. Esto ayuda que el modelo se estabilice en entrenamiento. Para el ancho y alto se usa exponencial (w = w_a · e^Δw) porque garantiza que las dimensiones siempre sean positivas ya que e^x nunca puede ser negativo o cero, sin importar qué valor prediga la red. En la función w_a es el ancho del anchor de referencia, Δw es el delta predicho por la red, y w es el ancho final de la caja.

### Respuesta 2.2.6

- Aunque Faster R-CNN representa una mejora significativa sigue tenienfo algunas limitaciones y seguramente no alcance a llegar hasta el rednimiento deseado. No es recomendable. Lo que pasa es que los 5fps se lograron en GPU, lo cual implica que al pasarlo a CPU el rendimiento puede ser considerablemente diferente. Una GPU tiene bastante más procesamiento dedicado para operaciones matemáticas en paralelo. Pero ademas de que el sistema seguramente sea mas lento, Faster R-CNN tiene cierta complejidad por que se compnone por 2 capas, primero por la RPN y también el detector. Ambas tienen su propio función de pérdida y comparten el backbone, lo que implica que se debe tener un finetuning muy detallado con los learning rate para que el modelo funcione. Otro punto es que con VGG16 se tiene un feature map único, lo cuál podría hacer dificil la detección de varios productos similares colocados cerca.

---
## Pregunta 2.3

El equipo de producto presenta dos propuestas para el detector de producción de VisorShelf:

| Dimensión | Propuesta A: Faster R-CNN + ResNet-50 + FPN | Propuesta B: YOLOv8n (nano) finetuned |
|---|---|---|
| Velocidad estimada | 8–12 FPS en GPU / ~1.5 FPS en CPU | 60–80 FPS en GPU / ~15 FPS en CPU |
| mAP@0.5 (referencia) | ~55 en COCO | ~37 en COCO |
| Tamaño del modelo | ~160 MB | ~6 MB |
| Manejo de objetos pequeños y densos | Alto (FPN multi-escala) | Moderado |
| Costo de fine-tuning | Moderado (pipeline de dos etapas) | Bajo (arquitectura simple) |

Responda en su reporte:

**7.** Tomando en cuenta las restricciones operativas de VisorShelf (hardware sin GPU, latencia < 500 ms, anaqueles densos), ¿cuál propuesta recomendaría y por qué? No se limite a comparar los números de la tabla; argumente la lógica de la decisión conectándola con las características técnicas de cada arquitectura.

**8.** Si VisorShelf logra instalar una GPU de gama media (RTX 3060) en las tiendas flagship, ¿cambiaría su recomendación? Explique cómo ese cambio de hardware altera el trade-off entre las dos propuestas.

**9.** ¿Qué riesgo técnico específico introduce hacer fine-tuning de Faster R-CNN con un dataset de anaqueles sin aplicar learning rate diferenciado entre el backbone y las capas nuevas? ¿Cómo se llama ese fenómeno y cómo lo mitigaría?

### Respuesta 2.3.7

- Ya que el negocio cuenta con varias restricciones, pero se busca una alternativa eficiente para el procesamiento en cpu, lo mejor sería utilizar YOLOv8n. Algo que le da una muy buena ventaja es que es un detector de una sola capa. Para este modelo procesa toda la imagen en un único forward pass sin etapa separada de propuestas de región, logrando ~15 FPS en CPU, lo cual son aprox 67ms/imagen. Mientras que faster R-CNN al tener dos etapas es significativamente mas lantes. Además el modelo es mucho menos pesado. Con respecto a mAP se ve que el valor es menor, pero podría realizarse cierto finetuning para reducir la diferencia, especialmente tomando en cuenta que el video que se busca evaluar constantemente es una estantería y la cámara es estática.

### Respuesta 2.3.8

- Al tener en mente el uso de GPU la decisión cambia ya que en ambos modelos la velocidad es suficiente para poder cumplir con el requisito de la velocidad. Entonces ya que el hardware deja de ser una limitante, la decisión se basa más en desempeño. Con una RTX 3060 disponible, cambiaría la recomendación a Faster R-CNN + ResNet50 + FPN. Al tener mucha más capacidad de procesamiento de video el diferenciador es la calidad de detección. R-CNN con FPN al ser más complejo y multicapa termina por ser el que tiene mejor precisión para detectar productos pequeños. Al tener un mejor hardware el tamaño del modelo y la complejidad del fine tuning ya no son tan limitantse.

### Respuesta 2.3.9

- El reisgo que aparece es el catastrophic forgetting. Lo que pasa es que si se aplica el mismo learning rate a todas las capas durante el fine-tuning está el riesgo que el backbone, la parte preentrenada en COCO con pesos ya calculados, se "olvide".  Si se aplica el mismo learning rate alto a todas las capas, los gradientes generados al principio por el error en SKU110K producen actualizaciones grandes en esos pesos, sobreescribiendo el conocimiento previo del backbone. La mitigación es aplicar learning rate diferenciado, osea usar un LR distinto por capa. Lo primero sería congelar el backbone en las primeras épocas para que sus pesos no cambien mientras la RPN y el clasificador aprenden a trabajar con imágenes de anaqueles. Una vez que esas capas funcionan razonablemente, se descongela el backbone con un LR muy bajo para que se adapte al entrenamiento nueveo de anaqueles sin perder los conocimientos base que ya tenía el modelo.